# Trees Homework

1. ## Data Preparation

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text

from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb

In [ ]:
data = "data/car_fuel_efficiency_2026.csv"

In [ ]:
df = pd.read_csv(data)

In [ ]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [ ]:
df.dtypes

,0
model_year,int64
origin,object
fuel_type,object
drivetrain,object
num_doors,int64
engine_displacement,int64
num_cylinders,int64
horsepower,float64
vehicle_weight,int64
acceleration,float64


In [ ]:
df.nunique()

,0
model_year,50
origin,3
fuel_type,3
drivetrain,3
num_doors,4
engine_displacement,127
num_cylinders,5
horsepower,153
vehicle_weight,176
acceleration,93


In [ ]:
# df.describe()

In [ ]:
df.isna().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,877
vehicle_weight,0
acceleration,264


In [ ]:
df = df.fillna(0)

#### Data Splitting

In [ ]:
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df, test_size=0.25, random_state=1)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

In [ ]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

#### Data One Hot Encoding

In [ ]:
dicts_train = df_train.to_dict(orient="records")
dicts_val = df_val.to_dict(orient="records")
dicts_test = df_test.to_dict(orient="records")

# dicts_train

In [ ]:
dv = DictVectorizer(sparse=False)
dv.fit(dicts_train)

dv.get_feature_names_out()

array(['acceleration', 'drivetrain=All-wheel drive',
       'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive',
       'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline',
       'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA',
       'vehicle_weight'], dtype=object)

In [ ]:
X_train = dv.transform(dicts_train)
X_val = dv.transform(dicts_val)

# X_train, X_val

#### Training

- Training A DecisionTreeRegressor model

In [ ]:
dtr = DecisionTreeRegressor(max_depth=1)

dtr.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1)

In [ ]:
export_text(dtr, feature_names=list(dv.get_feature_names_out()))

'|--- model_year <= 1995.50\n|   |--- value: [28.45]\n|--- model_year >  1995.50\n|   |--- value: [31.09]\n'

1. The original feature used for splitting the data is **model_year**

- Training A DecisionTreeRegressor model

In [ ]:
def train_predict_evaluate(X_train, y_train, X_val, y_val, n_estimators=10, max_depth=None):
  rfr = RandomForestRegressor(n_estimators=n_estimators, random_state=1, n_jobs=-1, max_depth=max_depth)
  rfr.fit(X_train, y_train)

  y_pred = rfr.predict(X_val)
  rmse = root_mean_squared_error(y_val, y_pred)
  return round(rmse, 3)

In [ ]:
rmse = train_predict_evaluate(X_train, y_train, X_val, y_val)
rmse

1.813

2. The RMSE of the model on the validation data is **1.813**

## Parameter Tuning

- Choosing the best nn_estimator

In [ ]:
n_estimators = [10, 50, 100, 150]

scores = {}

for n_estimator in n_estimators:
  rmse = train_predict_evaluate(X_train, y_train, X_val, y_val, n_estimators=n_estimator)
  scores[n_estimator] = rmse

scores

{10: 1.813, 50: 1.744, 100: 1.738, 150: 1.735}

3. The value gives that gives the lowest validation RMSE is : **150**

- Choosing the best nn_estimator

In [ ]:
n_estimators = [10, 50, 100, 150]
max_depths = [10, 15, 20, 25]

scores = {}
for max_depth in max_depths:
  for n_estimator in n_estimators:
    rmse = train_predict_evaluate(X_train, y_train, X_val, y_val, n_estimators=n_estimator, max_depth=max_depth)
    scores[max_depth] = rmse

scores

{10: 1.707, 15: 1.729, 20: 1.734, 25: 1.735}

4. The best max_depth is : **10**

## Feature Importance

In [ ]:
rfr = RandomForestRegressor(n_estimators=10,
  max_depth=20,
  random_state=1,
  n_jobs=-1)

rfr.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_estimators=10, n_jobs=-1, random_state=1)

In [ ]:
feature_importance = pd.DataFrame(rfr.feature_importances_, index=list(dv.get_feature_names_out()), columns=['importance'])
feature_importance.sort_values(by='importance', ascending=False)

,importance
model_year,0.304252
vehicle_weight,0.198357
fuel_type=Gasoline,0.159584
horsepower,0.081363
engine_displacement,0.065116
acceleration,0.057685
origin=USA,0.038670
num_doors,0.019276
drivetrain=Front-wheel drive,0.018043
fuel_type=Diesel,0.015946


5. What's the most important feature among vehicle_weight, horsepower, acceleration, engine_displacement is: **vehicle_weight**


Training a XGBoost model

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(dv.get_feature_names_out()))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(dv.get_feature_names_out()))

In [ ]:
def train_predict_evaluate_xgb(dtrain, dval, y_val, eta=0.3):
  xgb_params = {
    'eta': eta,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

  model = xgb.train(xgb_params, dtrain, num_boost_round=100)

  y_pred = model.predict(dval)

  return root_mean_squared_error(y_val, y_pred)

In [ ]:
# eta=0.3
train_predict_evaluate_xgb(dtrain, dval, y_val)

1.7813508581706494

In [ ]:
# eta=0.1
train_predict_evaluate_xgb(dtrain, dval, y_val, eta=0.1)

1.6815158734775855

6. The eta leads to the best RMSE score on the validation dataset is: **0.1**